In [ ]:
import os, sys, zipfile, subprocess
from pathlib import Path
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pytorch-msssim'])

PROJECT = Path('/content/drive/MyDrive/ImageHarmonization/iHarmony4_dataset_resize_256')
DATA = Path('/content/iharmony4_256')
DATA.mkdir(parents=True, exist_ok=True)

ARCHIVES = ['HCOCO', 'HAdobe5k', 'HFlickr', 'Hday2night']
for subset in ARCHIVES:
    dest = DATA / subset
    if not (dest / '.extracted_ok').exists():
        zip_path = PROJECT / f'{subset}_256.zip'
        assert zip_path.is_file(), f'Không tìm thấy {zip_path}'
        print(f'Đang giải nén {zip_path.name}...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(DATA)
        (dest / '.extracted_ok').touch()

print('Dữ liệu 256x256 sẵn sàng tại:', DATA)

Đang giải nén HCOCO_256.zip...
Đang giải nén HAdobe5k_256.zip...
Đang giải nén HFlickr_256.zip...
Đang giải nén Hday2night_256.zip...
Dữ liệu 256x256 sẵn sàng tại: /content/iharmony4_256


In [1]:
import json, random, hashlib, time, platform
from collections import defaultdict, Counter
import numpy as np
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import pil_to_tensor
from torchvision.utils import save_image
from pytorch_msssim import ssim
from tqdm.auto import tqdm

RUN = PROJECT / 'runs/unet_masked_l1_50ep_all4_bs32_seed42'
RUN.mkdir(parents=True, exist_ok=True)

CFG = dict(
    seed=42,
    epochs=50,
    checkpoint_milestone=30,
    size=256,
    batch_size=32,
    workers=2,
    base_channels=32,
    lr=3e-4,
    eta_min=1e-6,
    val_fraction=0.05,
    loss='masked_l1',
    architecture='unet_groupnorm_v1',
)

RESUME = True
DEVICE = torch.device('cuda')

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
torch.cuda.manual_seed_all(CFG['seed'])
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# 1. Lập chỉ mục & chia Train / Val / Test khớp 100% các run trước
EXTS = {'.jpg', '.jpeg', '.png'}
def make_index(folder):
    return {p.stem: p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in EXTS and not p.name.startswith('.')}

records = {'train': [], 'val': [], 'test': []}
official_counts = {}

for subset in ARCHIVES:
    subset_root = DATA / subset
    comp = make_index(subset_root / 'composite_images')
    mask = make_index(subset_root / 'masks')
    real = make_index(subset_root / 'real_images')

    subsets = {}
    for split in ('train', 'test'):
        txt = subset_root / f'{subset}_{split}.txt'
        rows = []
        for line in txt.read_text(encoding='utf-8-sig').splitlines():
            if not line.strip():
                continue
            stem = Path(line.strip().replace('\\', '/').split()[0]).stem
            parts = stem.split('_')
            source_id, mask_id = '_'.join(parts[:-2]), '_'.join(parts[:-1])
            rows.append(dict(
                subset=subset, stem=stem, group=f'{subset}/{source_id}',
                comp=str(comp[stem]), mask=str(mask[mask_id]), real=str(real[source_id]),
            ))
        subsets[split] = rows
        official_counts[f'{subset}_{split}'] = len(rows)

    groups = sorted({r['group'] for r in subsets['train']})
    random.Random(CFG['seed']).shuffle(groups)
    val_groups = set(groups[:max(1, round(len(groups) * CFG['val_fraction']))])

    for r in subsets['train']:
        records['val' if r['group'] in val_groups else 'train'].append(r)
    records['test'].extend(subsets['test'])

split_ids = {s: [f"{r['subset']}/{r['stem']}" for r in rows] for s, rows in records.items()}
fingerprint = hashlib.sha256(json.dumps(split_ids, sort_keys=True).encode()).hexdigest()
print(f"Train: {len(records['train'])} | Val: {len(records['val'])} | Test: {len(records['test'])}")

class HarmonyDataset(Dataset):
    def __init__(self, rows, train=False):
        self.rows, self.train = rows, train

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        target_size = (CFG['size'], CFG['size'])
        images = []
        for key, mode in [('comp', 'RGB'), ('mask', 'L'), ('real', 'RGB')]:
            with Image.open(r[key]) as im:
                im = im.convert(mode)
                if im.size != target_size:
                    resample = Image.Resampling.NEAREST if key == 'mask' else Image.Resampling.BILINEAR
                    im = im.resize(target_size, resample)
                images.append(pil_to_tensor(im).float() / 255.)
        c, m, y = images
        m_bin = (m >= 0.5).float()
        m = m_bin if m_bin.sum() > 0 else (m > 0).float()
        if self.train and torch.rand(()) < 0.5:
            c, m, y = [x.flip(-1) for x in (c, m, y)]
        return c, m, y, r['subset']

def loader(split, epoch=0):
    g = torch.Generator().manual_seed(CFG['seed'] + epoch)
    is_train = (split == 'train')
    return DataLoader(
        HarmonyDataset(records[split], is_train),
        batch_size=CFG['batch_size'],
        shuffle=is_train,
        num_workers=CFG['workers'],
        pin_memory=True,
        drop_last=False,
        persistent_workers=(CFG['workers'] > 0),
        generator=g,
    )

class Block(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False),
            nn.GroupNorm(8, cout),
            nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.GroupNorm(8, cout),
            nn.ReLU(inplace=True),
        )

class UNet(nn.Module):
    def __init__(self, b=32):
        super().__init__()
        self.enc = nn.ModuleList([Block(4, b), Block(b, b*2), Block(b*2, b*4), Block(b*4, b*8), Block(b*8, b*16)])
        self.up = nn.ModuleList([
            nn.ConvTranspose2d(b*16, b*8, 2, 2),
            nn.ConvTranspose2d(b*8, b*4, 2, 2),
            nn.ConvTranspose2d(b*4, b*2, 2, 2),
            nn.ConvTranspose2d(b*2, b, 2, 2),
        ])
        self.dec = nn.ModuleList([Block(b*16, b*8), Block(b*8, b*4), Block(b*4, b*2), Block(b*2, b)])
        self.head = nn.Conv2d(b, 3, 1)

    def forward(self, x):
        skips = []
        for i, block in enumerate(self.enc):
            x = block(x)
            if i < 4:
                skips.append(x)
                x = F.max_pool2d(x, 2)
        for up, block, skip in zip(self.up, self.dec, reversed(skips)):
            x = block(torch.cat([up(x), skip], 1))
        return self.head(x).sigmoid()

def masked_l1_loss(pred, target, mask, eps=1e-6):
    diff = torch.abs(pred - target) * mask
    per_img = diff.sum(dim=(1, 2, 3)) / (3.0 * mask.sum(dim=(1, 2, 3)).clamp_min(eps))
    return per_img.mean()

@torch.inference_mode()
def evaluate(loader_, save_to=None, include_ssim=False):
    model.eval()
    sums = defaultdict(Counter)
    for c, m, y, subsets in tqdm(loader_, desc='evaluate', leave=False):
        c, m, y = [x.to(DEVICE, non_blocking=True) for x in (c, m, y)]
        with torch.autocast('cuda', dtype=torch.float16):
            raw = model(torch.cat([c, m], 1))
        raw = raw.float()
        out = raw * m + c * (1 - m)
        err = (out - y).square()
        mse = err.mean((1, 2, 3))
        fmse = (err * m).sum((1, 2, 3)) / (3 * m.sum((1, 2, 3)).clamp_min(1.0))

        values = dict(
            raw_L1=(raw - y).abs().mean((1, 2, 3)),
            MSE=mse * 255**2,
            fMSE=fmse * 255**2,
            PSNR=-10 * torch.log10(mse.clamp_min(1e-12)),
            fPSNR=-10 * torch.log10(fmse.clamp_min(1e-12)),
        )
        if include_ssim:
            values['SSIM'] = ssim(out, y, data_range=1, size_average=False)

        for i, subset in enumerate(subsets):
            for label in ('all', subset):
                sums[label]['n'] += 1
                for k, v in values.items():
                    sums[label][k] += v[i].item()

        if save_to is not None:
            panels = torch.stack([c, m.expand_as(c), raw, out, y], 1).flatten(0, 1)
            save_image(panels[:30], str(save_to), nrow=5)
            save_to = None

    return {s: {k: (v if k == 'n' else v / d['n']) for k, v in d.items()} for s, d in sums.items()}

def atomic_save(obj, target):
    tmp = target.with_name(target.name + '.tmp')
    torch.save(obj, tmp)
    os.replace(tmp, target)

ModuleNotFoundError: No module named 'pytorch_msssim'

In [ ]:
model = UNet(CFG['base_channels']).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=CFG['lr'], betas=(0.9, 0.999))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'], eta_min=CFG['eta_min'])
scaler = torch.amp.GradScaler('cuda')

start_epoch, best, best_30, history = 0, float('inf'), float('inf'), []
if RESUME and (RUN / 'last.pt').exists():
    ck = torch.load(RUN / 'last.pt', map_location='cpu', weights_only=False)
    assert ck['cfg'] == CFG and ck['fingerprint'] == fingerprint, 'Config/data changed; use new RUN'
    model.load_state_dict(ck['model'])
    optimizer.load_state_dict(ck['optimizer'])
    for state in optimizer.state.values():
        for k, v in state.items():
            if torch.is_tensor(v):
                state[k] = v.to(DEVICE)
    scheduler.load_state_dict(ck['scheduler'])
    scaler.load_state_dict(ck['scaler'])
    start_epoch, best, history = ck['epoch'], ck['best'], ck['history']
    best_30 = ck.get('best_30', best)
    torch.set_rng_state(ck['torch_rng'])
    torch.cuda.set_rng_state_all(ck['cuda_rng'])
    print(f'Resume completed epoch {start_epoch} (best_50={best:.2f}, best_30={best_30:.2f})')

environment = dict(
    cfg=CFG,
    fingerprint=fingerprint,
    official_counts=official_counts,
    gpu=torch.cuda.get_device_name(),
    torch=torch.__version__,
    python=platform.python_version(),
)
(RUN / 'config.json').write_text(json.dumps(environment, indent=2))

for epoch in range(start_epoch, CFG['epochs']):
    torch.cuda.synchronize()
    tic = time.perf_counter()
    torch.cuda.reset_peak_memory_stats()
    model.train()
    total, count = 0., 0

    bar = tqdm(loader('train', epoch), desc=f"Masked L1 | Ep {epoch+1}/{CFG['epochs']}")
    for c, m, y, _ in bar:
        c, m, y = [x.to(DEVICE, non_blocking=True) for x in (c, m, y)]
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda', dtype=torch.float16):
            raw = model(torch.cat([c, m], 1))
            loss = masked_l1_loss(raw, y, m)

        if not torch.isfinite(loss):
            raise RuntimeError('Non-finite loss; inspect data/AMP')

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.)
        scaler.step(optimizer)
        scaler.update()

        bs = c.shape[0]
        total += loss.item() * bs
        count += bs
        bar.set_postfix(fL1=f'{total/count:.5f}', lr=f"{optimizer.param_groups[0]['lr']:.2e}")

    assert count == len(records['train']), f'Not all training samples were seen ({count} vs {len(records["train"])})'

    cur_ep = epoch + 1
    scores = evaluate(loader('val'), save_to=RUN / f'val_{cur_ep:03d}.png')
    cur_fmse = scores['all']['fMSE']

    improved = cur_fmse < best
    best = min(best, cur_fmse)

    lr_used = optimizer.param_groups[0]['lr']
    scheduler.step()
    torch.cuda.synchronize()

    row = dict(
        epoch=cur_ep,
        train_fL1=total / count,
        val=scores,
        lr=lr_used,
        seconds=time.perf_counter() - tic,
        peak_VRAM_GB=torch.cuda.max_memory_allocated() / 2**30,
    )
    history.append(row)

    payload = dict(model=model.state_dict(), cfg=CFG, epoch=cur_ep, fingerprint=fingerprint, val=scores)

    # 1. Lưu best riêng cho chặng 1 -> 30 (làm mốc pretrain chuẩn để tách nhánh B, C, D)
    if cur_ep <= CFG['checkpoint_milestone'] and cur_fmse < best_30:
        best_30 = cur_fmse
        atomic_save(payload, RUN / 'best_30ep.pt')

    # 2. Lưu best toàn cục trên cả 50 epochs (chính là kết quả hoàn chỉnh của Nhánh A)
    if improved:
        atomic_save(payload, RUN / 'best.pt')

    ck = dict(
        epoch=cur_ep,
        model=model.state_dict(),
        optimizer=optimizer.state_dict(),
        scheduler=scheduler.state_dict(),
        scaler=scaler.state_dict(),
        cfg=CFG,
        fingerprint=fingerprint,
        best=best,
        best_30=best_30,
        history=history,
        torch_rng=torch.get_rng_state(),
        cuda_rng=torch.cuda.get_rng_state_all(),
    )
    atomic_save(ck, RUN / 'last.pt')
    (RUN / 'history.json').write_text(json.dumps(history, indent=2))
    print(json.dumps(row, ensure_ascii=False))

print(f'Done Masked L1 (50 epochs)! Best 30ep fMSE: {best_30:.2f} | Best 50ep (Branch A) fMSE: {best:.2f}')

Masked L1 | Ep 1/50:   0%|          | 0/1952 [00:00<?, ?it/s]

evaluate:   0%|          | 0/103 [00:00<?, ?it/s]

{"epoch": 1, "train_fL1": 0.0843929531296308, "val": {"all": {"n": 3279, "raw_L1": 0.23681099060724042, "MSE": 82.86861321856314, "fMSE": 729.6607393990133, "PSNR": 33.655591185262345, "fPSNR": 21.598567254152847}, "HCOCO": {"n": 1924, "raw_L1": 0.23316427896848713, "MSE": 47.9403633371832, "fMSE": 700.9746717623515, "PSNR": 34.95944054260571, "fPSNR": 21.344510294543483}, "HAdobe5k": {"n": 974, "raw_L1": 0.25146678787726884, "MSE": 90.60489947081102, "fMSE": 600.2547519672088, "PSNR": 33.32676829841348, "fPSNR": 23.148923956148433}, "HFlickr": {"n": 368, "raw_L1": 0.2193005576365344, "MSE": 242.5355714702088, "fMSE": 1206.2041425290315, "PSNR": 27.878746823124263, "fPSNR": 18.91898936033249}, "Hday2night": {"n": 13, "raw_L1": 0.1741468356205867, "MSE": 152.82009979394766, "fMSE": 1180.8494732196514, "PSNR": 28.85144145672138, "fPSNR": 18.894169367276707}}, "lr": 0.0003, "seconds": 869.227776166, "peak_VRAM_GB": 6.012407302856445}


Masked L1 | Ep 2/50:   0%|          | 0/1952 [00:00<?, ?it/s]

evaluate:   0%|          | 0/103 [00:00<?, ?it/s]

{"epoch": 2, "train_fL1": 0.0774494729788335, "val": {"all": {"n": 3279, "raw_L1": 0.21851599532525762, "MSE": 79.46549777657057, "fMSE": 692.8036987154031, "PSNR": 33.94099326830297, "fPSNR": 21.93807191956367}, "HCOCO": {"n": 1924, "raw_L1": 0.21240898887016918, "MSE": 45.894466524684255, "fMSE": 662.5087757021374, "PSNR": 35.25363415244215, "fPSNR": 21.701707856818693}, "HAdobe5k": {"n": 974, "raw_L1": 0.23674703978415143, "MSE": 81.81146407072305, "fMSE": 553.5664235091552, "PSNR": 33.67933155525881, "fPSNR": 23.555915206854348}, "HFlickr": {"n": 368, "raw_L1": 0.2036571640966703, "MSE": 244.06065543190292, "fMSE": 1199.413323132888, "PSNR": 27.972510653993357, "fPSNR": 19.021187533502992}, "Hday2night": {"n": 13, "raw_L1": 0.17704622676739326, "MSE": 212.90203169675974, "fMSE": 1267.5880173903245, "PSNR": 28.228612459622898, "fPSNR": 18.276421840374287}}, "lr": 0.0002997049959000266, "seconds": 866.1520416349997, "peak_VRAM_GB": 6.012407302856445}


... (154 outputs left out)

Masked L1 | Ep 50/50:   0%|          | 0/1952 [00:00<?, ?it/s]

evaluate:   0%|          | 0/103 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d33ae8b13a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d33ae8b13a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__

    self._shutdown_workers()  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

{"epoch": 50, "train_fL1": 0.04573016220056419, "val": {"all": {"n": 3279, "raw_L1": 0.18311984751620064, "MSE": 39.44924833476271, "fMSE": 389.2281418821004, "PSNR": 36.094182289312, "fPSNR": 24.462769368175472}, "HCOCO": {"n": 1924, "raw_L1": 0.18096936985912293, "MSE": 22.011833580764563, "fMSE": 353.71565892146185, "PSNR": 37.3841678476631, "fPSNR": 24.24802695044361}, "HAdobe5k": {"n": 974, "raw_L1": 0.19192139854084783, "MSE": 39.317508312313215, "fMSE": 331.3201932172756, "PSNR": 35.93798347762967, "fPSNR": 26.199848830087962}, "HFlickr": {"n": 368, "raw_L1": 0.17218541443023994, "MSE": 126.24488491241051, "fMSE": 696.3334688725679, "PSNR": 30.038092670233354, "fPSNR": 21.20667023503262}, "Hday2night": {"n": 13, "raw_L1": 0.15147982365809953, "MSE": 173.0728258719811, "fMSE": 1290.274209829477, "PSNR": 28.313136761005108, "fPSNR": 18.26996143047626}}, "lr": 1.295004099973402e-06, "seconds": 869.1775687349946, "peak_VRAM_GB": 6.012407302856445}
Done Masked L1 (50 epochs)! Best 30

In [ ]:
RUN_TEST=True

In [ ]:
if RUN_TEST:
    ck = torch.load(RUN/'best.pt',map_location=DEVICE,weights_only=False)
    model.load_state_dict(ck['model'])
    test_scores = evaluate(loader('test'),save_to=RUN/'test_preview.png',include_ssim=True)
    (RUN/'test_metrics.json').write_text(json.dumps(test_scores,indent=2))
    print(json.dumps(test_scores,indent=2))
else:
    print('Test skipped. Set RUN_TEST=True only after finalizing the experiment.')



evaluate:   0%|          | 0/232 [00:00<?, ?it/s]

{
  "all": {
    "n": 7404,
    "raw_L1": 0.18175062785445356,
    "MSE": 41.66607108806385,
    "fMSE": 401.15732923377084,
    "PSNR": 35.9466533204918,
    "fPSNR": 24.163548158092283,
    "SSIM": 0.9844899420107719
  },
  "HCOCO": {
    "n": 4283,
    "raw_L1": 0.1804605342783495,
    "MSE": 26.79478604997287,
    "fMSE": 389.2966418986618,
    "PSNR": 36.83128595407839,
    "fPSNR": 23.79071955443709,
    "SSIM": 0.9888861556629864
  },
  "HAdobe5k": {
    "n": 2160,
    "raw_L1": 0.18635024088952276,
    "MSE": 43.736187594052815,
    "fMSE": 280.91971956204486,
    "PSNR": 35.965348217663944,
    "fPSNR": 26.127352466406645,
    "SSIM": 0.9802121067764582
  },
  "HFlickr": {
    "n": 828,
    "raw_L1": 0.1765746569737864,
    "MSE": 110.65730916604328,
    "fMSE": 703.3879070489303,
    "PSNR": 31.276810776208332,
    "fPSNR": 21.34266065053894,
    "SSIM": 0.9742098956724295
  },
  "Hday2night": {
    "n": 133,
    "raw_L1": 0.18081837612435334,
    "MSE": 57.43687587483485,
  